In [7]:
pip install numpy pandas scikit-learn tensorflow gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 75.8 MB/s eta 0:00:00


In [1]:
from google.colab import files
files.upload()  # Upload kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{\r\n  "username": "chaitanyadeore",\r\n  "key": "KGAT_907808ff718796b9e57c26d023bc468d"\r\n}'}

In [2]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)

import shutil
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')

os.chmod('/root/.kaggle/kaggle.json', 600)

In [3]:
!pip install kaggle

!kaggle datasets download -d clmentbisaillon/fake-and-real-news-dataset

Dataset URL: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
License(s): CC-BY-NC-SA-4.0
100% 41.0M/41.0M [00:04<00:00, 9.75MB/s]



In [4]:
import zipfile

with zipfile.ZipFile("fake-and-real-news-dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("data")

In [5]:
import pandas as pd

fake = pd.read_csv("data/Fake.csv")
real = pd.read_csv("data/True.csv")

fake['label'] = 0
real['label'] = 1

data = pd.concat([fake, real])
data = data[['text', 'label']]

data.to_csv("news.csv", index=False)

print("Dataset Shape:", data.shape)
data.head()

Dataset Shape: (44898, 2)


,text,label
0,Donald Trump just couldn t wish all Americans ...,0
1,House Intelligence Committee Chairman Devin Nu...,0
2,"On Friday, it was revealed that former Milwauk...",0
3,"On Christmas day, Donald Trump announced that ...",0
4,Pope Francis used his annual Christmas Day mes...,0


In [8]:
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

def clean_text(text):
    text = re.sub(r'[^a-zA-Z]', ' ', str(text))
    return text.lower()

data['text'] = data['text'].apply(clean_text)

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(data['text'])

sequences = tokenizer.texts_to_sequences(data['text'])
X = pad_sequences(sequences, maxlen=100)
from gensim.models import Word2Vec

sentences = [text.split() for text in data['text']]
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2)
y = data['label']

In [9]:
import numpy as np

embedding_matrix = np.zeros((5000, 100))

for word, i in tokenizer.word_index.items():
    if i < 5000 and word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential([
    Embedding(input_dim=5000,
              output_dim=100,
              weights=[embedding_matrix],
              input_length=100,
              trainable=False),
    LSTM(128),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 500,000 (1.91 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 500,000 (1.91 MB)

In [11]:
model.fit(X, y, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
562/562 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.9501 - loss: 0.1335 - val_accuracy: 0.9829 - val_loss: 0.0868
Epoch 2/5
562/562 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9747 - loss: 0.0710 - val_accuracy: 0.9853 - val_loss: 0.0558
Epoch 3/5
562/562 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9830 - loss: 0.0474 - val_accuracy: 0.9866 - val_loss: 0.0453
Epoch 4/5
562/562 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9910 - loss: 0.0284 - val_accuracy: 0.9748 - val_loss: 0.0728
Epoch 5/5
562/562 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9954 - loss: 0.0166 - val_accuracy: 0.9653 - val_loss: 0.1083


In [12]:
loss, acc = model.evaluate(X, y)
print("Accuracy:", acc)

1404/1404 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9914 - loss: 0.0299
Accuracy: 0.9913804531097412


In [13]:
def predict_news(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=100)
    pred = model.predict(padded)[0][0]

    return "Fake " if pred > 0.5 else "Real "

In [14]:
print(predict_news("Breaking: Government launches new scheme"))
print(predict_news("Aliens landed in Delhi yesterday"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
Real 
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Real 
